In [35]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# Initialize Local Spark Session
spark = SparkSession.builder \
    .appName("Week5AssignmentNotebook") \
    .master("local[*]") \
    .getOrCreate()
    
spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Active inside Jupyter!")

Spark Session Active inside Jupyter!


In [36]:
# Read the generated dataset
df = spark.read.csv("spark_assignment_dataset.csv", header=True, inferSchema=True)
print(f"Total rows loaded: {df.count()}")

Total rows loaded: 1250


In [37]:
# Remove duplicates based on user_id and transaction_date
df_deduped = df.dropDuplicates(["user_id", "transaction_date"])
df_deduped.show(10)

+-------+---------+--------------------+---+------------+----------+------+----------------+-----------+-------+--------+--------+-------------------+----------------+
|user_id| username|               email|age|subscription|      city|region|product_category|sale_amount|  price|store_id|  status|      raw_timestamp|transaction_date|
+-------+---------+--------------------+---+------------+----------+------+----------------+-----------+-------+--------+--------+-------------------+----------------+
|  U1000|   bob_45|bob.sharma4@yahoo...| 40|       Basic|    Ambala|  West|          Beauty|    1883.89|1883.89|   STR_5| Pending|2026-04-06 04:49:20|      2026-04-06|
|  U1001| clara_96|clara.smith9@gmai...| 59|       Basic|     Delhi| North|     Electronics|     524.19| 524.19|   STR_4|  Active|2026-02-15 13:34:48|      2026-02-15|
|  U1001| clara_96|clara.smith9@gmai...| 30|     Premium|    Mumbai|  West|     Electronics|     421.84| 421.84|   STR_4|Inactive|2026-03-02 06:27:05|      2026

In [38]:
# Fill null statuses
df_clean_status = df_deduped.na.fill({"status": "Unknown"})
df_clean_status.show(10)

+-------+---------+--------------------+---+------------+----------+------+----------------+-----------+-------+--------+--------+-------------------+----------------+
|user_id| username|               email|age|subscription|      city|region|product_category|sale_amount|  price|store_id|  status|      raw_timestamp|transaction_date|
+-------+---------+--------------------+---+------------+----------+------+----------------+-----------+-------+--------+--------+-------------------+----------------+
|  U1000|   bob_45|bob.sharma4@yahoo...| 40|       Basic|    Ambala|  West|          Beauty|    1883.89|1883.89|   STR_5| Pending|2026-04-06 04:49:20|      2026-04-06|
|  U1001| clara_96|clara.smith9@gmai...| 59|       Basic|     Delhi| North|     Electronics|     524.19| 524.19|   STR_4|  Active|2026-02-15 13:34:48|      2026-02-15|
|  U1001| clara_96|clara.smith9@gmai...| 30|     Premium|    Mumbai|  West|     Electronics|     421.84| 421.84|   STR_4|Inactive|2026-03-02 06:27:05|      2026

In [39]:
# Purge empty usernames or null emails
df_clean_users = df_clean_status.filter(
    F.col("email").isNotNull() & 
    (F.col("username") != "") & 
    F.col("username").isNotNull()
)
df_clean_users.show(10)

+-------+---------+--------------------+---+------------+----------+------+----------------+-----------+-------+--------+--------+-------------------+----------------+
|user_id| username|               email|age|subscription|      city|region|product_category|sale_amount|  price|store_id|  status|      raw_timestamp|transaction_date|
+-------+---------+--------------------+---+------------+----------+------+----------------+-----------+-------+--------+--------+-------------------+----------------+
|  U1000|   bob_45|bob.sharma4@yahoo...| 40|       Basic|    Ambala|  West|          Beauty|    1883.89|1883.89|   STR_5| Pending|2026-04-06 04:49:20|      2026-04-06|
|  U1001| clara_96|clara.smith9@gmai...| 59|       Basic|     Delhi| North|     Electronics|     524.19| 524.19|   STR_4|  Active|2026-02-15 13:34:48|      2026-02-15|
|  U1001| clara_96|clara.smith9@gmai...| 30|     Premium|    Mumbai|  West|     Electronics|     421.84| 421.84|   STR_4|Inactive|2026-03-02 06:27:05|      2026

In [40]:
# Filter for records where the region is strictly 'West'
df_filtered_region = df_clean_users.filter(F.col("region") == "West")
df_filtered_region.select("user_id", "username", "region").show(5)

+-------+---------+------+
|user_id| username|region|
+-------+---------+------+
|  U1000|   bob_45|  West|
|  U1001| clara_96|  West|
|  U1002|ananya_14|  West|
|  U1002|ananya_14|  West|
|  U1003| rahul_74|  West|
+-------+---------+------+
only showing top 5 rows


In [41]:
# Filter for a specific product category
df_filtered_category = df_clean_users.filter(F.col("product_category") == "Electronics")
df_filtered_category.select("user_id", "username", "product_category").show(5)

+-------+---------+----------------+
|user_id| username|product_category|
+-------+---------+----------------+
|  U1001| clara_96|     Electronics|
|  U1001| clara_96|     Electronics|
|  U1002|ananya_14|     Electronics|
|  U1003| rahul_74|     Electronics|
|  U1006|  john_29|     Electronics|
+-------+---------+----------------+
only showing top 5 rows


In [42]:
# Filtering Age Range
df_filtered_age = df_clean_users.filter(
    (F.col("age") >= 18) & 
    (F.col("age") <= 30)
)
df_filtered_age.select("user_id", "username", "age").show(5)

+-------+---------+---+
|user_id| username|age|
+-------+---------+---+
|  U1001| clara_96| 30|
|  U1003| rahul_74| 25|
|  U1004| rahul_63| 20|
|  U1004| rahul_63| 20|
|  U1005|ananya_10| 29|
+-------+---------+---+
only showing top 5 rows


In [43]:
# Combined Filters
df_targeted_audience = df_clean_users.filter(
    (F.col("region") == "West") &
    (F.col("age").between(18, 30)) &
    (F.col("product_category") == "Electronics")
)
print(f"Total target records found: {df_targeted_audience.count()}")
df_targeted_audience.select("user_id", "username", "age", "region", "product_category").show(10)

Total target records found: 29
+-------+----------+---+------+----------------+
|user_id|  username|age|region|product_category|
+-------+----------+---+------+----------------+
|  U1001|  clara_96| 30|  West|     Electronics|
|  U1029|  sneha_87| 30|  West|     Electronics|
|  U1040|  alice_39| 25|  West|     Electronics|
|  U1040|  alice_39| 25|  West|     Electronics|
|  U1060|  kabir_89| 29|  West|     Electronics|
|  U1076|   diya_57| 22|  West|     Electronics|
|  U1082|  david_58| 30|  West|     Electronics|
|  U1087|  ritik_43| 18|  West|     Electronics|
|  U1090|michael_34| 30|  West|     Electronics|
|  U1093|   john_48| 23|  West|     Electronics|
+-------+----------+---+------+----------------+
only showing top 10 rows


In [44]:
# Using count() to Calculate the total number of records in the cleaned dataset
total_records = df_clean_users.agg(F.count("user_id").alias("total_user_records"))
total_records.show()

# Using sum() to Calculate the total combined price of all products
total_sum = df_clean_users.agg(F.round(F.sum("price"), 2).alias("total_price_sum"))
total_sum.show()

# Using avg() to Calculate the average item price across the dataset
average_price = df_clean_users.agg(F.round(F.avg("price"), 2).alias("average_item_price"))
average_price.show()

# Using min() to Find the minimum product price in the dataset
minimum_price = df_clean_users.agg(F.round(F.min("price"), 2).alias("minimum_item_price"))
minimum_price.show()

# Using max() to Find the maximum product price in the dataset
maximum_price = df_clean_users.agg(F.round(F.max("price"), 2).alias("maximum_item_price"))
maximum_price.show()


+------------------+
|total_user_records|
+------------------+
|              1108|
+------------------+

+---------------+
|total_price_sum|
+---------------+
|     1072219.28|
+---------------+

+------------------+
|average_item_price|
+------------------+
|           1017.29|
+------------------+

+------------------+
|minimum_item_price|
+------------------+
|             59.03|
+------------------+

+------------------+
|maximum_item_price|
+------------------+
|           1999.45|
+------------------+



In [45]:
# Group data by city and compute the total record count per group
df_grouped_city = df_clean_users.groupBy("city") \
    .agg(F.count("user_id").alias("record_count")) \
    .orderBy(F.col("record_count").desc())

df_grouped_city.show()

# Applying condition on the aggregated results 
df_filtered_city_counts = df_grouped_city.filter(F.col("record_count") > 100)
df_filtered_city_counts.show()

+----------+------------+
|      city|record_count|
+----------+------------+
|    Ambala|         273|
|     Delhi|         246|
|   Mullana|         224|
|    Mumbai|         174|
| Bangalore|         101|
|   Kolkata|          46|
|Chandigarh|          44|
+----------+------------+

+---------+------------+
|     city|record_count|
+---------+------------+
|   Ambala|         273|
|    Delhi|         246|
|  Mullana|         224|
|   Mumbai|         174|
|Bangalore|         101|
+---------+------------+



In [46]:
from pyspark.sql.types import TimestampType

# Cast raw_timestamp from StringType to TimestampType
df_casted = df_clean_users.withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType()))
df_casted.select("raw_timestamp").printSchema()

# To Show sample rows to confirm formatting remains intact
df_casted.select("user_id", "raw_timestamp").show(3, truncate=False)

# Rename the casted column to event_time
df_renamed = df_casted.withColumnRenamed("raw_timestamp", "event_time")
df_renamed.select("event_time").printSchema()

# To Show the sample rows with the updated header name
df_renamed.select("user_id", "event_time").show(3, truncate=False)

root
 |-- raw_timestamp: timestamp (nullable = true)

+-------+-------------------+
|user_id|raw_timestamp      |
+-------+-------------------+
|U1000  |2026-04-06 04:49:20|
|U1001  |2026-02-15 13:34:48|
|U1001  |2026-03-02 06:27:05|
+-------+-------------------+
only showing top 3 rows
root
 |-- event_time: timestamp (nullable = true)

+-------+-------------------+
|user_id|event_time         |
+-------+-------------------+
|U1000  |2026-04-06 04:49:20|
|U1001  |2026-02-15 13:34:48|
|U1001  |2026-03-02 06:27:05|
+-------+-------------------+
only showing top 3 rows


In [47]:
# Audit the dataset for nulls and empty fields across key columns
df.select(
    F.count(F.when(F.col("price").isNull(), 1)).alias("null_prices"),
    F.count(F.when(F.col("email").isNull(), 1)).alias("null_emails"),
    F.count(F.when((F.col("username").isNull()) | (F.col("username") == ""), 1)).alias("corrupt_usernames")
).show()

# Standardize empty string anomalies into explicit Spark null values
df_standardized = df.withColumn(
    "username", 
    F.when(F.trim(F.col("username")) == "", F.lit(None)).otherwise(F.col("username"))
)
df_standardized.filter(F.col("username").isNull()).select("user_id", "username", "email").show(5)

# Fix financial missing metrics by providing a clean numeric baseline
df_clean_prices = df_standardized.na.fill({"price": 0.0})
df_clean_prices.select(
    F.count(F.when(F.col("price").isNull(), 1)).alias("remaining_null_prices")
).show()

+-----------+-----------+-----------------+
|null_prices|null_emails|corrupt_usernames|
+-----------+-----------+-----------------+
|         61|         37|               42|
+-----------+-----------+-----------------+

+-------+--------+--------------------+
|user_id|username|               email|
+-------+--------+--------------------+
|  U1124|    NULL|ananya.sharma7@ya...|
|  U1143|    NULL|priya.gupta8@yaho...|
|  U1263|    NULL|kabir.johnson3@ou...|
|  U1220|    NULL|ananya.johnson8@o...|
|  U1129|    NULL|priya.gupta1@gmai...|
+-------+--------+--------------------+
only showing top 5 rows
+---------------------+
|remaining_null_prices|
+---------------------+
|                    0|
+---------------------+



In [48]:
# Complete data processing pipeline
def run_analytics_pipeline(source_df):
    """
    Executes a production-grade end-to-end data processing pipeline:
    1. Removes row duplicates based on tracking tracking columns (user_id & transaction_date).
    2. Corrects empty username fields by converting them to explicit nulls.
    3. Handles missing financial data by establishing a 0.0 baseline for null prices.
    4. Groups the clean records by store_id to summarize total revenue.
    """
    # Step A: Remove Duplicates (Wide Transformation boundary)
    deduplicated_df = source_df.dropDuplicates(["user_id", "transaction_date"])
    
    # Step B: Standardize Inconsistent Text Fields (Narrow Transformation)
    standardized_df = deduplicated_df.withColumn(
        "username",
        F.when(F.trim(F.col("username")) == "", F.lit(None)).otherwise(F.col("username"))
    )
    
    # Step C: Fill Missing Financial Metrics (Narrow Transformation)
    cleaned_df = standardized_df.na.fill({"price": 0.0})
    
    # Step D: Group and Aggregate Final Results (Wide Transformation & Shuffle)
    pipeline_output = cleaned_df.groupBy("store_id") \
        .agg(F.round(F.sum("price"), 2).alias("total_revenue")) \
        .orderBy("store_id")
        
    return pipeline_output

# Execute the pipeline on the original raw DataFrame 'df'
final_pipeline_results = run_analytics_pipeline(df)

# Result: Display the aggregated metrics table
final_pipeline_results.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|   STR_1|    246541.04|
|   STR_2|    213646.09|
|   STR_3|    200059.23|
|   STR_4|    252898.35|
|   STR_5|    233283.32|
+--------+-------------+

